# India Unemployment Rate Analysis

This notebook analyzes unemployment rate data for India, focusing on key trends, the Covid-19 impact, seasonal patterns, and policy-relevant insights.

In [ ]:
# Section 1: Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

## Load and Inspect Datasets

In [ ]:
# Section 2: Load and inspect datasets
base_path = Path(r'c:\Users\Mudassar\Desktop\CodeAlpha-Data Science\Unemployment_Analysis')
file1 = base_path / 'Unemployment in India.csv'
file2 = base_path / 'Unemployment_Rate_upto_11_2020.csv'

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

print('Dataset 1 shape:', df1.shape)
print('Dataset 2 shape:', df2.shape)
print('\nDataset 1 columns:', df1.columns.tolist())
print('Dataset 2 columns:', df2.columns.tolist())

In [ ]:
# Section 2 continued: Display dataset samples
print('\nDataset 1 sample:')
print(df1.head().to_string(index=False))
print('\nDataset 2 sample:')
print(df2.head().to_string(index=False))

## Clean and Merge Data

In [ ]:
# Section 3: Clean and merge data

def clean_columns(df):
    df = df.copy()
    df.columns = [col.strip() for col in df.columns]
    return df

df1 = clean_columns(df1)
df2 = clean_columns(df2)

if 'Region.1' in df2.columns:
    df2 = df2.rename(columns={'Region.1': 'Zone'})

if 'Area' not in df2.columns:
    df2['Area'] = np.nan

common_cols = [
    'Region',
    'Date',
    'Frequency',
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)',
    'Area',
]

for df in [df1, df2]:
    for col in common_cols:
        if col not in df.columns:
            df[col] = np.nan

merged = pd.concat([df1[common_cols], df2[common_cols]], ignore_index=True)
print('Merged shape:', merged.shape)

In [ ]:
print('\nMerged data sample:')
print(merged.head().to_string(index=False))

## Convert Dates and Fix Data Types

In [ ]:
# Section 4: Convert dates and numeric columns
merged = merged.copy()
merged['Date'] = pd.to_datetime(merged['Date'], dayfirst=True, errors='coerce')
merged['Estimated Unemployment Rate (%)'] = pd.to_numeric(merged['Estimated Unemployment Rate (%)'], errors='coerce')
merged['Estimated Employed'] = pd.to_numeric(merged['Estimated Employed'], errors='coerce')
merged['Estimated Labour Participation Rate (%)'] = pd.to_numeric(merged['Estimated Labour Participation Rate (%)'], errors='coerce')

merged['Year'] = merged['Date'].dt.year
merged['Month'] = merged['Date'].dt.month
merged['MonthName'] = merged['Date'].dt.month_name()
merged['YearMonth'] = merged['Date'].dt.to_period('M')

print('Converted data types and added date features.')
print(merged[['Date', 'Year', 'Month', 'MonthName', 'YearMonth']].head().to_string(index=False))

## Handle Missing Values and Duplicates

In [ ]:
# Section 5: Clean missing values and duplicates
missing_before = merged.isna().sum()
print('Missing values before cleaning:\n', missing_before)

deduped = merged.drop_duplicates(subset=['Region', 'Date', 'Estimated Unemployment Rate (%)', 'Area'])
print('Duplicates removed:', merged.shape[0] - deduped.shape[0])

cleaned = deduped.dropna(subset=['Region', 'Date', 'Estimated Unemployment Rate (%)']).copy()
cleaned['Region'] = cleaned['Region'].astype(str).str.strip()
cleaned = cleaned[cleaned['Region'] != '']
cleaned = cleaned.reset_index(drop=True)

print('Shape after cleaning:', cleaned.shape)

## Explore Unemployment Rates by Region and Area

In [ ]:
# Section 6: Region summary statistics
summary_region = (
    cleaned.groupby('Region')['Estimated Unemployment Rate (%)']
           .agg(['mean', 'median', 'std', 'count'])
           .sort_values('mean', ascending=False)
           .reset_index()
)
summary_region.head(15)

In [ ]:
# Section 6 continued: Area summary statistics
summary_area = (
    cleaned.groupby('Area')['Estimated Unemployment Rate (%)']
           .agg(['mean', 'median', 'std', 'count'])
           .reset_index()
           .sort_values('mean', ascending=False)
)
summary_area

## Visualize Trends Before and During Covid-19

In [ ]:
# Section 7: National trend visualization
cleaned['CovidPeriod'] = np.where(cleaned['Date'] >= pd.Timestamp('2020-03-01'), 'During Covid', 'Pre Covid')

national_ts = (
    cleaned.groupby('YearMonth')['Estimated Unemployment Rate (%)']
           .mean()
           .reset_index()
)
national_ts['YearMonth'] = national_ts['YearMonth'].dt.to_timestamp()

plt.figure(figsize=(14, 6))
sns.lineplot(data=national_ts, x='YearMonth', y='Estimated Unemployment Rate (%)', marker='o')
plt.axvspan(pd.Timestamp('2020-03-01'), national_ts['YearMonth'].max(), alpha=0.2, color='red')
plt.title('India Average Unemployment Rate Over Time')
plt.xlabel('Month')
plt.ylabel('Average Unemployment Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Section 7 continued: Compare representative regions
regions_to_plot = ['Uttar Pradesh', 'Maharashtra', 'Bihar', 'Delhi']
region_ts = (
    cleaned[cleaned['Region'].isin(regions_to_plot)]
           .groupby(['Region', 'YearMonth'])['Estimated Unemployment Rate (%)']
           .mean()
           .reset_index()
)
region_ts['YearMonth'] = region_ts['YearMonth'].dt.to_timestamp()

plt.figure(figsize=(14, 8))
sns.lineplot(data=region_ts, x='YearMonth', y='Estimated Unemployment Rate (%)', hue='Region', marker='o')
plt.title('Unemployment Rate Trends for Selected Regions')
plt.xlabel('Month')
plt.ylabel('Unemployment Rate (%)')
plt.legend(title='Region')
plt.tight_layout()
plt.show()

## Detect Seasonal and Monthly Patterns

In [ ]:
# Section 8: Monthly pattern analysis
monthly_pattern = (
    cleaned.groupby('Month')['Estimated Unemployment Rate (%)']
           .mean()
           .reindex(range(1, 13))
           .reset_index()
)

plt.figure(figsize=(12, 5))
sns.barplot(data=monthly_pattern, x='Month', y='Estimated Unemployment Rate (%)', palette='viridis')
plt.title('Average Unemployment Rate by Month')
plt.xlabel('Month')
plt.ylabel('Average Unemployment Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Section 8 continued: Rolling average trend smoothing
national_ts['RollingMean'] = national_ts['Estimated Unemployment Rate (%)'].rolling(window=3, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(national_ts['YearMonth'], national_ts['Estimated Unemployment Rate (%)'], marker='o', label='Monthly Average')
plt.plot(national_ts['YearMonth'], national_ts['RollingMean'], marker='o', linestyle='--', label='3-Month Rolling Mean')
plt.title('National Unemployment Rate with Rolling Average')
plt.xlabel('Month')
plt.ylabel('Average Unemployment Rate (%)')
plt.legend()
plt.tight_layout()
plt.show()

## Summarize Policy-Relevant Insights

In [ ]:
# Section 9: Summarize Policy-Relevant Insights
covid_summary = (
    cleaned.groupby('CovidPeriod')['Estimated Unemployment Rate (%)']
           .agg(['mean', 'median', 'std', 'count'])
           .reset_index()
)

top_regions_covid = (
    cleaned[cleaned['Date'] >= pd.Timestamp('2020-03-01')]
           .groupby('Region')['Estimated Unemployment Rate (%)']
           .mean()
           .sort_values(ascending=False)
           .head(10)
           .reset_index()
)

print('Covid period unemployment summary:\n', covid_summary)
print('\nTop 10 regions by average unemployment rate during Covid period:\n', top_regions_covid)

### Key Insights

- Covid-19 caused a sharp rise in unemployment rates from March 2020 onward.
- States such as Bihar, Delhi, and Uttar Pradesh showed the highest average unemployment during the Covid period.
- Monthly seasonality is visible, with higher unemployment around the lockdown months of April and May 2020.
- Policy action should prioritize support for high-risk regions and strengthen labor market monitoring.

### Policy Recommendations

1. Focus relief programs on regions with the largest Covid-era unemployment spikes.
2. Strengthen rural and urban employment support to reduce seasonal vulnerability.
3. Use rolling averages and monthly monitoring to spot sudden labor market shocks.
4. Invest in retraining and livelihood diversification for affected workers.